In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
import sklearn
np.__version__, pd.__version__, sklearn.__version__

('2.4.3', '3.0.1', '1.8.0')

### Q1. 

In [ ]:
df = pd.read_csv("set_04_data.csv")
df.head(2)

In [5]:
ser_1665 = df.loc[df["user"] == 1665, "city"]
ser_1665.nunique()

2

In [ ]:
df_user_city = df.groupby("user")["city"].nunique().reset_index()
df_user_city.head()

In [ ]:
ser_id_city_GE2 = df_user_city.loc[df_user_city["city"] >= 2, "user"]
ser_id_city_GE2.head() # city가 2개 이상인 ID 목록

In [ ]:
df_q1 = df.loc[~df["user"].isin(ser_id_city_GE2), ]
df["user"].nunique(), df_q1["user"].nunique()

In [11]:
df_q1 = df_q1.loc[(df_q1["job"] == 20) & (df_q1["gender"] == "M"), ]
len(df_q1), df_q1["user"].nunique()

(1869, 94)

In [ ]:
df_q1_c5 = df_q1.loc[df_q1["prod_cat1"] == 5, ]
len(df_q1_c5)

In [14]:
round(len(df_q1_c5) / len(df_q1), 3)

0.295

### Q2. 

In [15]:
df = pd.read_csv("set_04_data.csv")
df.head(2)

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
0,3451,P00375436,F,26-35,11,A,0,20,0,0,595
1,3829,P00003242,F,26-35,1,B,0,8,15,0,5872


In [19]:
ls_col_cat = ["prod_cat1", "prod_cat2", "prod_cat3"]
df_u1 = df.loc[df["user"] == 1, ls_col_cat]
df_u1 = df_u1.sort_values(ls_col_cat).reset_index(drop = True)
df_u1.drop_duplicates().shape

(9, 3)

In [22]:
df_u1["prod_cat1"] = df_u1["prod_cat1"].astype("str")
df_u1["prod_cat2"] = df_u1["prod_cat2"].astype("str")
df_u1["prod_cat3"] = df_u1["prod_cat3"].astype("str")
df_u1["prod_cat"] = df_u1["prod_cat1"] + "-" + df_u1["prod_cat2"] + "-" + df_u1["prod_cat3"]

In [ ]:
df_u1.head(2)

In [26]:
df_u1["prod_cat"].nunique()

9

In [ ]:
df_q2 = df.loc[df["age_group"] == "26-35", ["user", "marital"] + ls_col_cat].reset_index(drop = True)
df_q2["prod_cat1"] = df_q2["prod_cat1"].astype("str")
df_q2["prod_cat2"] = df_q2["prod_cat2"].astype("str")
df_q2["prod_cat3"] = df_q2["prod_cat3"].astype("str")
df_q2["prod_cat" ] = df_q2["prod_cat1"] + "-" + df_q2["prod_cat2"] + "-" + df_q2["prod_cat3"]
df_q2.head(2)

In [35]:
df_q2_g1 = df_q2.groupby("user")["prod_cat"].nunique().reset_index()
df_q2_g1.head()

In [ ]:
df_q2_g2 = df_q2[["user", "marital"]].drop_duplicates().tail()
df_q2_g_join = pd.merge(df_q2_g1, df_q2_g2, on = "user", how = "inner")
df_q2_g_join.head(2)

In [ ]:
df_q2_g3 = df_q2.groupby(["user", "marital"])["prod_cat"].nunique().reset_index()
df_q2_g3.head(2)

In [38]:
stat_m0 = df_q2_g3.loc[df_q2_g3["marital"] == 0, "prod_cat"].mean()
stat_m1 = df_q2_g3.loc[df_q2_g3["marital"] == 1, "prod_cat"].mean()
stat_m0, stat_m1

(np.float64(21.147342995169083), np.float64(21.121287128712872))

In [39]:
round(abs(stat_m0 - stat_m1), 2)

np.float64(0.03)

### Q3.

In [ ]:
df = pd.read_csv("set_04_data.csv")
df.head(2)

In [45]:
# 단계 1)
ser_city = df.groupby("user")["city"].nunique()
ser_city_GE2 = ser_city[ser_city >= 2].index
# ser_city_GE2
df_q3 = df.loc[~df["user"].isin(ser_city_GE2), ]
df_q3["user"].nunique()

3142

In [ ]:
# 단계 2)
ls_col_step2 = ["user", "gender", "age_group", "job", "city", "marital"]
df_q3_user = df_q3[ls_col_step2].drop_duplicates()
df_q3_user.head(2)

In [ ]:
# 단계 3)
df_q3_g1 = df_q3.groupby("user")[["prod", "purchase"]].agg({"prod": "nunique",
                                                            "purchase": "sum"}).reset_index()
df_q3_g1.head()

In [49]:
# 단계 4)
df_q3_join = pd.merge(df_q3_user, df_q3_g1, on = "user")
df_q3_join.head(2)

,user,gender,age_group,job,city,marital,prod,purchase
0,5368,F,26-35,17,C,0,14,128695
1,5840,F,0-17,0,C,0,46,319395


In [50]:
len(df), len(df_q3_join)

(200000, 3142)

In [53]:
# 단계 5)
df_q3_join["gender"] = df_q3_join["gender"].replace({"M": 1, "F": 0})

In [55]:
df_q3_join["age_group"].sort_values().unique()

<StringArray>
['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']
Length: 7, dtype: str

In [ ]:
ser_u = df_q3_join["age_group"].sort_values().unique()
ser_repl = pd.Series(range(len(ser_u)), index = ser_u)
ser_repl.to_dict() # 굳이 몰라도 됨.

In [58]:
df_q3_join["age_group"] = df_q3_join["age_group"].replace(ser_repl)

In [60]:
df_q3_join["gender"] = df_q3_join["gender"].astype("int") # 최신버전만 필요한 코드
df_q3_join["age_group"] = df_q3_join["age_group"].astype("int") # 최신버전만 필요한 코드

In [ ]:
df_q3_join.head(2)

In [ ]:
# 단계 6)
# df_q3_join_dum = pd.get_dummies(df_q3_join, columns = ["job", "city"]) # 시험버전
df_q3_join_dum = pd.get_dummies(df_q3_join, columns = ["job", "city"], dtype = "int") # 최신버전
df_q3_join_dum.head(1)

In [64]:
df_q3_join_dum = df_q3_join_dum.drop(columns = "user")

In [ ]:
# 단계 7)
arr_model = MinMaxScaler().fit_transform(df_q3_join_dum)
arr_model[:1, ]

In [ ]:
# 단계 8)
model_kmeans = KMeans(n_clusters = 7, random_state = 123)
model_kmeans.fit(arr_model)

In [69]:
round(silhouette_score(arr_model, labels = model_kmeans.labels_), 2)

0.17